# 🔁 Generator–Critic Refinement Loop

## Learning Objectives
In this notebook, you will learn:
1. **The loop** - A generator agent writes, a critic agent evaluates, and the draft is revised until it passes or a cap is reached
2. **Acceptance criteria** - Split the criteria into rules code can check and judgements only a model can make
3. **A typed critic** - Make the critic return a validated `Review` with `create_agent(..., response_format=ToolStrategy(Review))`
4. **Two exits** - Wire the loop with `add_conditional_edges`: accept, revise, or stop at the revision cap
5. **The stop-at-limit path** - Prove the cap ends the loop, with a stand-in generator and no LLM

## Where this fits
This is pattern 4 of [Five Practical Agent Patterns](../../../../docs/source/08/five-practical-agent-patterns.md), also called **evaluator–optimizer**. It reuses two ideas from `01_sequential_and_conditional_workflows.ipynb`: a typed agent decision and a conditional edge. The new part is an edge that goes **back** to an earlier node, which turns the workflow into a loop.

**Flow:** `Generator → critic → accepted result OR feedback → generator`

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads the API keys from the project-root `.env` and sets `RECURSION_LIMIT`, the step cap every agent run passes.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and imports
# ============================================================================
import operator
import warnings
from typing import Annotated, Literal, TypedDict

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Every agent run passes this cap on LangGraph super-steps
RECURSION_LIMIT = 25

print(f"✅ Environment ready (recursion_limit = {RECURSION_LIMIT})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model shared by every agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# ASK: run a prebuilt agent on one message and return its final reply
# ============================================================================
def ask(agent, text: str) -> str:
    """Send one user message to a create_agent agent; return the text of its last message."""
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, {"recursion_limit": RECURSION_LIMIT})
    return result["messages"][-1].content


print("✅ ask() ready")

---
## 📏 Part 1: Acceptance Criteria First

A refinement loop is only as good as its definition of "done". The task: a product description for a solar phone charger. Its criteria fall into two groups:

| Criterion | Checked by | Why |
|---|---|---|
| At most `MAX_WORDS` words | code | counting is exact and free |
| Mentions the price `₹2,499` | code | a substring check |
| No exclamation marks | code | a substring check |
| Names a concrete hiking or camping use | critic agent | needs judgement |
| States a clear benefit and sounds professional | critic agent | needs judgement |

The critic step runs the **code checks first**. If any fails, the feedback is the list of failures and no model is called. Only a draft that passes every code check goes to the critic agent. This keeps the critic honest on the rules code can verify, and saves a model call on every draft that breaks one.

In [ ]:
# ============================================================================
# ACCEPTANCE CRITERIA: rules for code, judgements for the critic
# ============================================================================
TASK = "Write a product description for a foldable solar phone charger for hikers."
MAX_WORDS = 50
PRICE = "₹2,499"
MAX_REVISIONS = 3  # the cap: the loop writes at most this many drafts

REQUIREMENTS = f"""- At most {MAX_WORDS} words
- Mention the price {PRICE}
- No exclamation marks
- Name a concrete hiking or camping use
- State one clear benefit in a professional tone"""


def code_checks(draft: str) -> list[str]:
    """Return one problem per rule the draft breaks; an empty list means every code check passes."""
    problems = []
    words = len(draft.split())
    if words > MAX_WORDS:
        problems.append(f"Too long: {words} words; the limit is {MAX_WORDS}.")
    if PRICE not in draft:
        problems.append(f"Missing the exact price {PRICE}.")
    if "!" in draft:
        problems.append("Remove every exclamation mark.")
    return problems


print(code_checks("Amazing charger!"))  # breaks two rules
print(code_checks(f"A foldable charger for hikers that keeps a phone alive on multi-day treks. {PRICE}."))

---
## ✍️ Part 2: The Generator and the Typed Critic

- **`generator`** is a plain `create_agent` agent. On a revision, its prompt includes the previous draft and the feedback to address.
- **`critic`** is a `create_agent` agent with `response_format=ToolStrategy(Review)`. Its reply is a validated `Review` in `result["structured_response"]`, so the loop branches on a real `bool`, never on parsed text.

In [ ]:
# ============================================================================
# GENERATOR AND CRITIC: two prebuilt agents
# ============================================================================
class Review(BaseModel):
    """The critic's verdict on one draft."""

    approved: bool = Field(description="True only if the draft meets every requirement")
    feedback: str = Field(description="Specific, actionable changes; empty when approved")


generator = create_agent(
    llm,
    system_prompt=(
        "You write short product descriptions that follow every requirement given. "
        "When you receive feedback, revise the previous draft to address all of it. "
        "Return only the description."
    ),
    name="generator",
)
critic = create_agent(
    llm,
    system_prompt=(
        "You are a strict marketing editor. Judge the draft against every requirement. "
        "Approve only if all are met; otherwise list the exact changes needed."
    ),
    response_format=ToolStrategy(Review),
    name="critic",
)

print("✅ Agents ready: generator, critic")

---
## 🔁 Part 3: Wire the Loop

```
START → generate → critique ─┬─ approved ───────────────→ END   (accept)
              ↑              ├─ revision cap reached ───→ END   (stop)
              └── feedback ──┘  otherwise                       (revise)
```

The state keeps the current `draft`, the latest `feedback`, a `revision` counter, and a `history` list with one entry per critique (the `operator.add` reducer appends). `should_continue` is plain code with three exits. The `stop` exit is what guarantees the loop ends even if the critic never approves.

`build_refinement_loop(generate_node)` takes the generate node as an argument, so Part 5 can swap in a stand-in generator without touching the loop.

In [ ]:
# ============================================================================
# REFINEMENT GRAPH: generate -> critique -> accept / revise / stop
# ============================================================================
class RefineState(TypedDict):
    task: str
    draft: str
    feedback: str  # the latest feedback; empty before the first critique
    approved: bool
    revision: int  # drafts written so far
    history: Annotated[list[dict], operator.add]  # one entry per critique


def generate(state: RefineState) -> dict:
    prompt = f"Task: {state['task']}\nRequirements:\n{REQUIREMENTS}"
    if state.get("feedback"):
        prompt += f"\n\nPrevious draft:\n{state['draft']}\n\nFeedback to address:\n{state['feedback']}"
    return {"draft": ask(generator, prompt), "revision": state.get("revision", 0) + 1}


def critique(state: RefineState) -> dict:
    problems = code_checks(state["draft"])
    if problems:  # a rule code can check failed: no model call needed
        approved, feedback, by = False, "\n".join(problems), "code"
    else:
        result = critic.invoke(
            {"messages": [HumanMessage(content=f"Requirements:\n{REQUIREMENTS}\n\nDraft:\n{state['draft']}")]},
            {"recursion_limit": RECURSION_LIMIT},
        )
        review = result["structured_response"]  # a Review instance
        approved, feedback, by = review.approved, review.feedback, "critic"
    entry = {"revision": state["revision"], "draft": state["draft"], "approved": approved, "by": by, "feedback": feedback}
    return {"approved": approved, "feedback": feedback, "history": [entry]}


def should_continue(state: RefineState) -> Literal["accept", "stop", "revise"]:
    if state["approved"]:
        return "accept"
    if state["revision"] >= MAX_REVISIONS:
        return "stop"
    return "revise"


def build_refinement_loop(generate_node):
    builder = StateGraph(RefineState)
    builder.add_node("generate", generate_node)
    builder.add_node("critique", critique)
    builder.add_edge(START, "generate")
    builder.add_edge("generate", "critique")
    builder.add_conditional_edges(
        "critique", should_continue, {"accept": END, "stop": END, "revise": "generate"}
    )
    return builder.compile()


refine_app = build_refinement_loop(generate)
print(refine_app.get_graph().draw_mermaid())

In [ ]:
# ============================================================================
# SHOW_HISTORY: print every draft and the verdict on it
# ============================================================================
def show_history(result: dict):
    for entry in result["history"]:
        verdict = "✅ approved" if entry["approved"] else f"🔁 rejected by {entry['by']}"
        print(f"--- draft {entry['revision']} ({len(entry['draft'].split())} words): {verdict}")
        print(entry["draft"])
        if entry["feedback"]:
            print(f"feedback: {entry['feedback']}")
        print()
    outcome = "accepted" if result["approved"] else f"stopped at the cap ({MAX_REVISIONS} drafts)"
    print(f"OUTCOME: {outcome} after {result['revision']} draft(s)")


print("✅ show_history() ready")

---
## ▶️ Part 4: Run the Loop

In [ ]:
# ============================================================================
# RUN: GENERATE AND REFINE UNTIL ACCEPTED OR CAPPED
# ============================================================================
result = refine_app.invoke({"task": TASK, "history": []}, {"recursion_limit": RECURSION_LIMIT})
show_history(result)

ended_correctly = result["approved"] or result["revision"] == MAX_REVISIONS
print(f"\n{'✅' if ended_correctly else '❌'} the loop ended on approval or at the cap")
if result["approved"]:
    print(f"{'✅' if not code_checks(result['draft']) else '❌'} the accepted draft passes every code check")

Read the history from top to bottom. A draft rejected **by code** never reached the critic agent. Each later draft should fix the feedback printed above it. If the run ended at the cap, the last draft is the best one the loop produced, not an approved one: a caller should treat it that way.

---
## 🛑 Part 5: Prove the Stop-at-Limit Path

The run above probably ended with `accepted`, so the `stop` exit has not been exercised. A stand-in generator that always writes the same over-long draft with an exclamation mark fails the code checks every time. The loop must then end at `MAX_REVISIONS` drafts and never call the critic agent. No LLM is involved, so this check gives the same answer on every run.

In [ ]:
# ============================================================================
# STOP-AT-LIMIT PROOF: a stand-in generator that never improves (no LLM)
# ============================================================================
BAD_DRAFT = "This charger is amazing! " * 20  # 80 words, exclamation marks, no price


def stuck_generate(state: RefineState) -> dict:
    return {"draft": BAD_DRAFT, "revision": state.get("revision", 0) + 1}


stuck_result = build_refinement_loop(stuck_generate).invoke({"task": TASK, "history": []})

print(f"drafts written:   {stuck_result['revision']}")
print(f"critique entries: {len(stuck_result['history'])}")
print(f"{'✅' if not stuck_result['approved'] else '❌'} never approved")
print(f"{'✅' if stuck_result['revision'] == MAX_REVISIONS else '❌'} stopped at MAX_REVISIONS = {MAX_REVISIONS}")
print(f"{'✅' if all(e['by'] == 'code' for e in stuck_result['history']) else '❌'} the critic agent was never called")

---
## 📝 Summary

In this notebook, we learned:

### 1. The loop
- A conditional edge from `critique` back to `generate` turns a workflow into a refinement loop.

### 2. Acceptance criteria
- Write the criteria before building the loop. Check in code what code can check; send only judgement calls to the critic agent.

### 3. A typed critic
- `create_agent(..., response_format=ToolStrategy(Review))` returns a validated `Review`, so the loop branches on `approved: bool`.

### 4. Two exits
- `should_continue` returns `accept`, `revise` or `stop`. The cap (`MAX_REVISIONS`) guarantees the loop ends even when the critic never approves.

### 5. The stop-at-limit path
- A stand-in generator proved the cap without an LLM: three drafts, no approval, no critic call.

### Next Steps
- **`05_handoffs_with_swarm.ipynb`**: let agents transfer the conversation itself to each other with `langgraph-swarm`